## 1. Setup and Installation

Install required packages and configure your environment.

In [ ]:
# Install dependencies (run once)
import sys
!{sys.executable} -m pip install -q python-dotenv google-genai

print("✅ Dependencies installed")

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"✅ Project root: {project_root}")

## 2. Configure Google Gemini API

**Required:** Get your free API key at https://aistudio.google.com/app/apikey

In [ ]:
# Option 1: Load from .env file (recommended)
load_dotenv()

# Option 2: Set directly (not recommended - don't commit this!)
# os.environ['GOOGLE_API_KEY'] = 'your-key-here'

# Verify API key
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if GOOGLE_API_KEY:
    print("✅ Google Gemini API configured")
    print(f"   Key: {GOOGLE_API_KEY[:8]}...{GOOGLE_API_KEY[-4:]}")
else:
    print("❌ GOOGLE_API_KEY not found!")
    print()
    print("Please set your API key:")
    print("  1. Get free key: https://aistudio.google.com/app/apikey")
    print("  2. Create a .env file with: GOOGLE_API_KEY=your-key-here")
    print("     OR set environment: export GOOGLE_API_KEY='your-key-here'")

## 3. Define ODD Specification

Describe your robot's Operational Design Domain in natural language. The AI will parse this into a formal specification.

**Pipeline stages:**
1. **ODD Spec Agent** - Parse natural language ODD → formal specification
2. **Perception Agents** - Analyze camera + LiDAR images (loop + summary)
3. **Motion Agents** - Detect motion from IMU data (loop + summary)
4. **Collision Agents** - Assess collision risk (loop + summary)
5. **COD Classifier** - Determine current operating domain
6. **ODD Compliance** - Compare COD vs ODD, detect violations
7. **Report Generator** - Create comprehensive analysis report

In [ ]:
# Natural Language ODD Description
# The AI will parse this and create a formal specification

odd_description = """
The Unitree Go2 quadruped robot is designed for indoor navigation in office environments.

OPERATIONAL CONSTRAINTS:

1. Environment Type:
   - Designed for: indoor_office, indoor_corridor
   - Prohibited: outdoor environments, staircases, unstructured terrain

2. Lighting Conditions:
   - Required: bright or dim lighting (adequate visibility)
   - Prohibited: dark environments (requires vision sensors)

3. Terrain Type:
   - Designed for: smooth_floor (tile, hardwood, low-pile carpet)
   - Prohibited: rough or very_rough terrain, stairs, slopes >10°

4. Speed Range:
   - Normal operation: 0.0 to 1.5 m/s
   - Physical limit: 2.5 m/s (emergency only)

5. Obstacle Density:
   - Acceptable: low to moderate obstacles (0.0 to 0.6 normalized)
   - Boundary: 0.6 to 0.8 (crowded but navigable)
   - Prohibited: >0.8 (too cluttered for safe navigation)

6. Traversability:
   - Required: navigable space (0.5 to 1.0 score)
   - Boundary: 0.3 to 0.5 (challenging but possible)
   - Prohibited: <0.3 (impassable or unsafe)

7. Collision Risk:
   - Acceptable: low risk (0.0 to 0.3 likelihood)
   - Boundary: 0.3 to 0.5 (caution required)
   - Prohibited: >0.5 (high risk, stop immediately)

8. Platform Stability:
   - Required: stable platform (roll/pitch <15°)
   - Boundary: 15° to 20° (unstable but recoverable)
   - Prohibited: >20° (tip-over risk)
"""

print(f"✅ ODD specification defined ({len(odd_description)} characters)")
print("\n💡 TIP: Customize this for different robot types:")
print("   • Outdoor delivery robots (weather, GPS, terrain)")
print("   • Aerial drones (altitude, wind speed, battery)")
print("   • Warehouse AMRs (floor type, shelf proximity)")
print("   • Autonomous vehicles (road type, traffic, visibility)")

## 4. Select Scenario to Analyze

Choose which preprocessed dataset you want to analyze. Scenarios are stored in `data/processed/runs/`.

**Available scenarios:**
- `sim_run_test` - Small test dataset (2 windows)
- `demo_run` - Demo dataset for quick testing
- Your own scenarios (after running `extract_windows.py`)

In [ ]:
# List available scenarios
data_dir = project_root / "data" / "processed" / "runs"

if data_dir.exists():
    scenarios = [d.name for d in data_dir.iterdir() if d.is_dir()]
    print("📁 Available scenarios:")
    for i, scenario in enumerate(scenarios, 1):
        scenario_path = data_dir / scenario
        window_count = len(list(scenario_path.glob("motion_*.json")))
        print(f"   {i}. {scenario:20s} ({window_count:2d} windows)")
else:
    print("⚠️ Data directory not found!")
    print(f"   Expected: {data_dir}")
    print("   Run: python scripts/extract_windows.py first")
    scenarios = []

print()

# ========== CONFIGURATION: Select scenario ==========
SCENARIO_NAME = "sim_run_test"

if scenarios and SCENARIO_NAME in scenarios:
    SCENARIO_PATH = str((data_dir / SCENARIO_NAME).absolute())
    window_count = len(list((data_dir / SCENARIO_NAME).glob("motion_*.json")))
    print(f"✅ Selected: {SCENARIO_NAME}")
    print(f"   Windows: {window_count}")
    print(f"   Path: {SCENARIO_PATH}")
else:
    print(f"❌ Scenario '{SCENARIO_NAME}' not found!")
    if scenarios:
        print(f"   Available: {', '.join(scenarios)}")
    SCENARIO_PATH = None

## 5.1. Configure Models

Select which Gemini models to use for each agent. Balance cost vs quality:

**Available models:**
- `gemini-2.0-flash-lite` - Fast, cost-effective
- `gemini-2.5-pro` - Best quality (higher cost, recommended for complex tasks)

**Model selection strategy:**
- Use `gemini-2.5-pro` for challenging tasks: perception, collision analysis, ODD spec parsing
- Use `gemini-2.0-flash-lite` for simpler tasks: motion detection, report generation

In [ ]:
# ========== CONFIGURATION: Model Selection ==========
# Customize which models to use for each agent
# Options: "gemini-2.0-flash-lite", "gemini-2.5-pro"

MODEL_PERCEPTION = "gemini-2.5-pro"       # Camera + LiDAR analysis (complex vision tasks)
MODEL_MOTION = "gemini-2.0-flash-lite"    # IMU motion detection (straightforward)
MODEL_COLLISION = "gemini-2.5-pro"        # Collision risk assessment (complex reasoning)
MODEL_ODD_SPEC = "gemini-2.5-pro"         # ODD specification parsing (complex NLP)
MODEL_COD = "gemini-2.0-flash-lite"       # COD classification + compliance
MODEL_REPORT = "gemini-2.0-flash-lite"    # Final report generation

print("🔧 Model Configuration:")
print(f"   Perception:  {MODEL_PERCEPTION}")
print(f"   Motion:      {MODEL_MOTION}")
print(f"   Collision:   {MODEL_COLLISION}")
print(f"   ODD Spec:    {MODEL_ODD_SPEC}")
print(f"   COD/Comply:  {MODEL_COD}")
print(f"   Report:      {MODEL_REPORT}")
print()
print("💡 TIP: Use 'gemini-2.5-pro' for complex tasks, 'flash-lite' for simple ones")
print("   See ../docs/MODEL_SELECTION_GUIDE.md for detailed recommendations")

## 5.2. Run Workflow

Execute the complete 10-agent pipeline. This will:
- Parse the ODD specification
- Analyze all time windows (perception, motion, collision)
- Classify the operating domain
- Check ODD compliance
- Generate final report

In [ ]:
from google.genai import Client
from odd_agents import run_odd_workflow

# Verify configuration
if not GOOGLE_API_KEY:
    print("❌ Google API Key not set! Please run cell 2.")
elif not SCENARIO_PATH:
    print("❌ Scenario not selected! Please run cell 4.")
else:
    # Create client
    genai_client = Client(api_key=GOOGLE_API_KEY)
    
    print("🚀 Starting ODD analysis workflow...")
    print(f"   Scenario: {SCENARIO_NAME}")
    print(f"   Path: {SCENARIO_PATH}")
    print(f"   ODD description: {len(odd_description)} characters")
    print()
    print("⏳ This may take 2-3 minutes...")
    print()

    try:
        # Run the workflow with configured models
        result = await run_odd_workflow(
            scenario_path=SCENARIO_PATH,
            genai_client=genai_client,
            api_key=GOOGLE_API_KEY,
            nl_odd_description=odd_description,
            model_perception=MODEL_PERCEPTION,
            model_motion=MODEL_MOTION,
            model_collision=MODEL_COLLISION,
            model_odd_spec=MODEL_ODD_SPEC,
            model_cod=MODEL_COD,
            model_report=MODEL_REPORT,
        )
        
        if result:
            print()
            print("=" * 80)
            print("✅ ANALYSIS COMPLETE!")
            print("=" * 80)
            
            # Quick summary
            report = result.get('report', {})
            metadata = report.get('scenario_metadata', {})
            compliance_data = result.get('full_analysis', {}).get('odd_compliance', {}).get('odd_compliance', {})
            
            print(f"\n📊 Summary:")
            print(f"   • Windows analyzed: {metadata.get('total_windows_analyzed', 'N/A')}")
            print(f"   • Data source: {metadata.get('data_source', 'N/A')} (confidence: {metadata.get('data_source_confidence', 'N/A')})")
            print(f"   • ODD compliance: {compliance_data.get('overall_compliance', 'N/A')}")
            print(f"   • Violations: {len(compliance_data.get('violations', []))}")
            print(f"   • Warnings: {len(compliance_data.get('warnings', []))}")
            
        else:
            print()
            print("❌ Workflow failed - no results generated")
            result = None
            
    except Exception as e:
        print()
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        result = None

## 6. View Executive Summary

High-level findings and key insights from the analysis.

In [ ]:
if result:
    report = result['report']
    
    print("=" * 80)
    print("EXECUTIVE SUMMARY")
    print("=" * 80)
    print()
    print(report.get('executive_summary', 'N/A'))
    print()
    
    print("=" * 80)
    print("KEY FINDINGS")
    print("=" * 80)
    for i, finding in enumerate(report.get('key_findings', []), 1):
        print(f"\n{i}. {finding}")
    
    print()
    print("=" * 80)
    print("RECOMMENDATIONS")
    print("=" * 80)
    for i, rec in enumerate(report.get('recommendations', []), 1):
        print(f"\n{i}. {rec}")
else:
    print("⚠️ No results available. Please run the workflow first (cell 5).")

## 7. ODD Compliance Details

Detailed breakdown of compliance status across all analyzed dimensions.

In [ ]:
if result:
    compliance_data = result['full_analysis']['odd_compliance']['odd_compliance']
    
    print("=" * 80)
    print("ODD COMPLIANCE ANALYSIS")
    print("=" * 80)
    print()
    print(f"Overall Status: {compliance_data.get('overall_compliance', 'N/A')}")
    print(f"Compliance Summary: {compliance_data.get('compliance_summary', 'N/A')}")
    print()
    
    violations = compliance_data.get('violations', [])
    warnings = compliance_data.get('warnings', [])
    
    if violations:
        print("❌ VIOLATIONS DETECTED:")
        print("-" * 80)
        for violation in violations:
            print(f"  • {violation}")
        print()
    else:
        print("✅ NO VIOLATIONS DETECTED")
        print()
    
    if warnings:
        print("⚠️  WARNINGS:")
        print("-" * 80)
        for warning in warnings:
            print(f"  • {warning}")
        print()
    
    print("📊 CATEGORICAL COMPLIANCE:")
    print("-" * 80)
    for axis, status in compliance_data.get('categorical_compliance', {}).items():
        icon = "✅" if status == "IN_ODD" else ("⚠️" if status == "ODD_BOUNDARY" else "❌")
        print(f"{icon} {axis:30s} → {status}")
    
    print()
    print("📏 NUMERIC COMPLIANCE:")
    print("-" * 80)
    for axis, status in compliance_data.get('numeric_compliance', {}).items():
        icon = "✅" if status == "IN_ODD" else ("⚠️" if status == "ODD_BOUNDARY" else "❌")
        print(f"{icon} {axis:30s} → {status}")
else:
    print("⚠️ No results available. Please run the workflow first (cell 5).")

## 8. Visualize Results

Plot collision risk and motion detection over time.

In [ ]:
if result:
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Extract data
    full_analysis = result['full_analysis']
    collision_events = full_analysis['collision']['collision_events']
    motion_windows = full_analysis['motion']['per_window_motion']
    
    if collision_events and motion_windows:
        # Prepare data
        windows = [w['window_id'] for w in collision_events]
        risk_levels = [w['collision_risk_level'] for w in collision_events]
        risk_scores = [w.get('risk_confidence', 0) for w in collision_events]
        motion_detected = [w['motion_detected'] for w in motion_windows]
        motion_types = [w['motion_type'] for w in motion_windows]
        
        # Create figure
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
        
        # Plot 1: Collision risk
        color_map = {'none': 'green', 'low': 'lightgreen', 'medium': 'orange', 'high': 'red', 'critical': 'darkred'}
        colors = [color_map.get(level, 'gray') for level in risk_levels]
        
        ax1.bar(range(len(windows)), risk_scores, color=colors, alpha=0.7, edgecolor='black')
        ax1.axhline(y=0.3, color='orange', linestyle='--', linewidth=1, label='Caution (0.3)')
        ax1.axhline(y=0.5, color='red', linestyle='--', linewidth=1, label='High Risk (0.5)')
        ax1.set_ylabel('Risk Confidence', fontsize=12, fontweight='bold')
        ax1.set_title('Collision Risk Assessment by Window', fontsize=14, fontweight='bold')
        ax1.legend(loc='upper right')
        ax1.grid(axis='y', alpha=0.3)
        ax1.set_ylim(0, 1.0)
        
        # Plot 2: Motion detection
        motion_values = [1 if m else 0 for m in motion_detected]
        motion_colors = ['red' if m else 'lightgray' for m in motion_detected]
        ax2.bar(range(len(windows)), motion_values, color=motion_colors, alpha=0.7, edgecolor='black')
        
        # Add motion type labels
        for i, (detected, mtype) in enumerate(zip(motion_detected, motion_types)):
            if detected:
                ax2.text(i, 0.5, mtype, ha='center', va='center', fontsize=9, fontweight='bold')
        
        ax2.set_ylabel('Motion Detected', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Window ID', fontsize=12, fontweight='bold')
        ax2.set_title('Motion Detection (IMU-based)', fontsize=14, fontweight='bold')
        ax2.set_xticks(range(len(windows)))
        ax2.set_xticklabels(windows)
        ax2.set_yticks([0, 1])
        ax2.set_yticklabels(['No Motion', 'Motion'])
        ax2.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print statistics
        motion_stats = full_analysis['motion']['overall_stats']
        
        print()
        print("📊 STATISTICS:")
        print("-" * 80)
        print(f"Motion - Detection rate: {motion_stats.get('motion_detection_rate', 0)*100:.1f}%")
        print(f"Motion - Max accel: {motion_stats.get('max_horizontal_accel_mps2', 0):.4f} m/s²")
        print(f"Motion - Max angular vel: {motion_stats.get('max_angular_velocity_radps', 0):.4f} rad/s")
        print(f"Motion - Assessment: {motion_stats.get('overall_assessment', 'N/A')}")
    else:
        print("⚠️ No per-window data available for visualization")
else:
    print("⚠️ No results available. Please run the workflow first (cell 5).")

## 9. Export Results

Save the complete analysis report to JSON for further processing or sharing.

In [ ]:
if result:
    # Report is already saved by run_odd_workflow()
    output_path = Path(SCENARIO_PATH) / "odd_analysis_report.json"
    
    if output_path.exists():
        file_size = output_path.stat().st_size / 1024
        print(f"✅ Results saved to:")
        print(f"   {output_path}")
        print(f"   Size: {file_size:.1f} KB")
        
        # Optionally, save to a different location
        # import json
        # custom_path = project_root / "my_analysis_results.json"
        # with open(custom_path, 'w') as f:
        #     json.dump(result, f, indent=2)
        # print(f"\n📁 Also saved to: {custom_path}")
    else:
        print("⚠️ Report file not found at expected location")
    
    # Access the result directly
    print()
    print("💡 TIP: Access results programmatically:")
    print("   result['report']                                # Human-readable report")
    print("   result['full_analysis']['perception']           # Perception data")
    print("   result['full_analysis']['motion']               # Motion data")
    print("   result['full_analysis']['collision']            # Collision data")
    print("   result['full_analysis']['odd_spec']             # Parsed ODD spec")
    print("   result['full_analysis']['cod_classification']   # COD classification")
    print("   result['full_analysis']['odd_compliance']       # Compliance analysis")
else:
    print("⚠️ No results to export. Please run the workflow first (cell 5).")

## Next Steps

### 🎯 Customize Your Analysis

1. **Modify the ODD** (cell 3) to match your robot's design constraints
2. **Test different scenarios** (cell 4) to analyze various datasets
3. **Adjust parameters** by editing the ODD description

### 📊 Advanced Usage

- **Compare scenarios**: Run the workflow on multiple datasets and compare results
- **Parameter sensitivity**: Test how ODD threshold changes affect compliance
- **Custom visualizations**: Access `result` dictionary to create your own plots

### 🔍 Dive Deeper

- **View agent implementations**: `odd_agents/agents/` directory
- **Understand the workflow**: `odd_agents/workflow.py`
- **Read documentation**: `docs/` directory
- **Factory pattern explanation**: `docs/FACTORY_PATTERN.md`

### 🚀 Production Deployment

To use this in production:
1. Preprocess ROS2 bags: `python scripts/extract_windows.py`
2. Run analysis: `python scripts/odd_workflow.py`
3. Automate with CI/CD for continuous monitoring

---

**Questions?** Check the project documentation or explore the source code!